<a href="https://colab.research.google.com/github/jabri62018/Zx_RieOS_v1.2/blob/Zx_RieOS_v1.2/Zx_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:

import mpmath as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, zipfile, os
from google.colab import files

DPS = 80
mp.mp.dps = DPS
xp = 21.0

def Zx(t):
    x = mp.mpc('0.5', str(t))
    return mp.exp(-x/xp) * mp.exp(-5*mp.log(x)) * mp.log(x) * mp.sin(2*mp.pi/x)

def find_brackets(step=0.05, t_start=0.001, t_end=60, max_roots=5):
    mp.mp.dps = DPS - 20
    t_vals = np.arange(t_start, t_end, step)
    f_vals = [float(mp.im(Zx(t))) for t in t_vals]
    brackets = []
    for j in range(len(f_vals)-1):
        if not np.isnan(f_vals[j]) and not np.isnan(f_vals[j+1]):
            if f_vals[j] * f_vals[j+1] < 0:
                brackets.append((t_vals[j], t_vals[j+1]))
    return brackets[:max_roots]

def refine_roots(brackets, dps=DPS):
    mp.mp.dps = dps
    roots = []
    for t_min, t_max in brackets:
        root = mp.findroot(lambda tt: mp.im(Zx(tt)), (t_min, t_max),
                           tol=mp.mpf(f'1e-{dps-15}'), maxsteps=800)
        if abs(float(mp.im(Zx(root)))) < 1e-50:
            roots.append(float(root))
    return sorted(roots)

# 1. Compute roots and match constants
brackets = find_brackets()
roots = refine_roots(brackets)

CONSTANTS = {
    'G': 6.67430e-11,
    'alpha': 7.2973525693e-3,
    'pi': np.pi,
    'e': np.e,
    'phi': (1+np.sqrt(5))/2
}
MEANING = {'G':'Gravity', 'alpha':'Fine Structure', 'pi':'Electromagnetism',
           'e':'Black Holes', 'phi':'Golden Ratio'}

rows = []
available_consts = CONSTANTS.copy()

for i, g in enumerate(roots, 1):
    mp.mp.dps = DPS
    h = mp.mpf('1e-15')
    t = mp.mpf(g)

    z = Zx(t)
    zp = (Zx(t+h) - Zx(t-h)) / (2*h)
    zpp = (Zx(t+h) - 2*z + Zx(t-h)) / (h*h)
    zppp = (Zx(t+2*h) - 2*Zx(t+h) + 2*Zx(t-h) - Zx(t-2*h)) / (2*h)
    C_calc = 0.5 * g**2 * mp.re(zppp / z)

    logC = mp.log10(abs(C_calc) + mp.mpf('1e-300'))
    diffs = {k: abs(logC - mp.log10(abs(mp.mpf(v)) + mp.mpf('1e-300')))
             for k,v in available_consts.items()}
    matched = min(diffs, key=diffs.get)
    logdiff = float(diffs[matched])
    del available_consts[matched]

    rows.append({
        'Root #': i,
        'Gamma': float(g),
        'C_calc': float(C_calc),
        'Matched Constant': matched,
        'Constant Value': CONSTANTS[matched],
        'Meaning': MEANING[matched],
        'Log Diff': logdiff
    })

df = pd.DataFrame(rows)
df.to_csv('Zx5_roots_match.csv', index=False, float_format='%.15e')
print("Saved: Zx5_roots_match.csv")

# 2. Save plot as PNG
mp.mp.dps = DPS - 20
t_vals = np.linspace(0.001, 60, 8000)
f_vals = [float(mp.im(Zx(t))) for t in t_vals]

plt.figure(figsize=(13, 6))
plt.plot(t_vals, f_vals, linewidth=0.7, color='royalblue', label='Im[Zx(t)]')
plt.yscale('symlog', linthresh=1e-25)
plt.axhline(0, color='black', linewidth=1, alpha=0.6)

for i, row in df.iterrows():
    r = float(row['Gamma'])
    plt.plot(r, 0, 'ro', markersize=8)
    plt.text(r, 1e-15, f"R{row['Root #']}-{row['Matched Constant']}",
             ha='center', va='bottom', fontsize=10, color='darkred', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.xlabel('t', fontsize=12)
plt.ylabel('Im[Zx(t)] - Symlog Scale', fontsize=12)
plt.title(f'First 5 Zeros of Zx_5 Mapped to Physical Constants | {DPS}-digit precision',
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('Zx5_roots_plot.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: Zx5_roots_plot.png")

# 3. Build.ipynb file
nb = {
 "cells": [
  {
   "cell_type": "markdown",
   "source": ["# Zx_5 Root Analysis\n",
              "Computes first 5 zeros of Im[Zx(t)], matches each to a unique physical constant,",
              " and plots with symlog scale."]
  },
  {
   "cell_type": "code",
   "source": [open(__file__).read() if '__file__' in globals() else "# Code is in this cell"]
  }
 ],
 "metadata": {"language_info": {"name": "python"}},
 "nbformat": 4,
 "nbformat_minor": 5
}

with open('Zx5_analysis.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=2)
print("Saved: Zx5_analysis.ipynb")

# 4. Zip everything and download
zip_name = 'Zx5_results.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('Zx5_roots_match.csv')
    zipf.write('Zx5_roots_plot.png')
    zipf.write('Zx5_analysis.ipynb')

print(f"\nDone! Created {zip_name}")
files.download(zip_name)

Saved: Zx5_roots_match.csv
Saved: Zx5_roots_plot.png
Saved: Zx5_analysis.ipynb

Done! Created Zx5_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>